# 08 — When to Roll: Offensive vs Defensive Rolls by Delta

Rolling is **position management**: instead of holding a short put until a fixed
exit, we watch its delta every day and re-strike when it drifts too far from
target. Delta is the natural trigger because it *is* the market's estimate of
how threatened the strike is.

| Roll type | Trigger | Mechanics | Purpose |
|---|---|---|---|
| **Defensive** | \|delta\| rises to threshold (e.g. 0.50) — strike is being tested | Buy back the tested put (realize the leg loss), sell a new put at the original target delta in a later expiration (out **and down**) | Avoid the full stop-loss; keep collecting premium while the strike resets away from danger |
| **Offensive** | \|delta\| falls to threshold (e.g. 0.12) — trade won early | Buy back the near-worthless put (lock the profit), sell a new one at target delta | Redeploy capital: a 5-delta short put earns almost nothing per day of risk |

A **campaign** = the chain of legs from first entry until management stops
(the held leg reaches `exit_dte`, or `max_rolls` is exhausted). Campaign P&L
sums all legs — that's what we compare against the no-roll baseline **on
identical entry dates**.

This needs `lab.rolling` (a purpose-built simulator) because optopsy models a
trade as one entry + one exit; it cannot track a specific contract day by day.

**Caveats before believing anything below:** triggers fire at EOD closes (a
0.50-delta breach at 11am fires at the close print), fills are mid with no
transaction costs, and SPX is European-style so there is no early assignment
to model.

In [ ]:
import sys
sys.path.insert(0, "../src")
%load_ext autoreload
%autoreload 2

import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (11, 4)
pd.set_option("display.width", 160)

### Load chains and set the study window
The roll simulator tracks individual contracts daily, so it only needs the put
side of the chains. Widen the window to 2010 for the full study — this default
keeps the notebook fast.

In [ ]:
from lab.backtest import load_chains
from lab.rolling import RollConfig, simulate_campaigns, compare_variants, summarize_campaigns

START, END = "2016-01-01", "2023-12-31"   # widen to 2010-01-01 for the full study
chains = load_chains(START, END)
print(f"{len(chains):,} rows loaded")

### The four variants, on identical monthly entries
- **no-roll**: sell ~30-delta, 45 DTE; close at 21 DTE. The control.
- **defensive**: same, but roll out+down whenever \|delta\| ≥ 0.50
- **offensive**: same, but roll up whenever \|delta\| ≤ 0.12
- **both**: both triggers armed

Same entry dates everywhere, so any difference is purely the roll policy.

In [ ]:
summary, camps = compare_variants(chains, defensive=0.50, offensive=0.12)
summary.round(2)

### Campaign equity curves
Cumulative campaign P&L by exit date. Watch how the variants separate in
drawdown periods (2018 Q4, 2020 COVID, 2022 bear) — that's where roll policy
actually matters. In calm years all four lines should track closely.

In [ ]:
ax = None
for name, camp in camps.items():
    cum = camp.sort_values("exit_date").set_index("exit_date")["total_pnl"].cumsum()
    ax = cum.plot(ax=ax, label=name)
ax.legend(); ax.set_title("Cumulative campaign P&L by roll policy"); ax.set_ylabel("$");

### When to roll defensively: sweep the trigger delta
The core "when" question. A low trigger (0.40) rolls early and often — paying
the bid/ask churn and realizing many small losses. A high trigger (0.60) waits
so long the leg loss is nearly a full stop-out. Look for the sweet spot, and —
as in notebook 04 — distrust a spike whose neighbors are bad.

In [ ]:
from dataclasses import replace

rows = []
for thr in (0.40, 0.45, 0.50, 0.55, 0.60):
    _, camp = simulate_campaigns(chains, RollConfig(defensive_delta=thr))
    rows.append(summarize_campaigns(camp, f"def@{thr}"))
def_sweep = pd.DataFrame(rows).set_index("variant")
def_sweep.round(2)

In [ ]:
def_sweep[["total_pnl"]].plot.bar(legend=False, title="Total P&L by defensive trigger delta")
plt.ylabel("$");

### When to roll offensively: sweep the trigger delta
Offensive rolls trade a locked-in win for fresh risk. A high trigger (0.20)
re-strikes constantly; a low one (0.08) almost never fires. The question is
whether redeploying early beats simply holding the winner to `exit_dte` —
often it only increases time-in-market without improving P&L per day.

In [ ]:
rows = []
for thr in (0.08, 0.12, 0.15, 0.20):
    _, camp = simulate_campaigns(chains, RollConfig(offensive_delta=thr))
    rows.append(summarize_campaigns(camp, f"off@{thr}"))
off_sweep = pd.DataFrame(rows).set_index("variant")
off_sweep.round(2)

### Anatomy of the rolls
Leg-level view of the `both` variant. `close_reason` tells each leg's story:
- **defensive_roll** legs should show losses (we bought back a tested put) —
  the question is whether the *following* legs earn it back
- **offensive_roll** legs should be near-full winners
- **exit / expiry** legs are the natural campaign ends

The roll-count histogram shows how often management actually engages: most
campaigns should end with 0–1 rolls; frequent 3–4 roll campaigns mean the
triggers are too tight.

In [ ]:
cfg_both = RollConfig(defensive_delta=0.50, offensive_delta=0.12)
legs, camp_both = simulate_campaigns(chains, cfg_both)
print(legs.groupby("close_reason")["pnl"].agg(["count", "mean", "sum"]).round(1))
(camp_both.n_defensive + camp_both.n_offensive).value_counts().sort_index().plot.bar(
    title="Rolls per campaign", xlabel="rolls", ylabel="campaigns");

### Does defensive rolling pay in every vol regime?
Join each campaign's entry date to the feature matrix and bucket by VIX rank.
Expectation: defensive rolling earns its keep when vol is high/rising (strikes
get tested), and mostly costs churn in the calm buckets. If it *loses* in the
calm buckets, consider arming it conditionally — e.g. only when vix_rank > 0.5.

In [ ]:
from lab.backtest import load_features

feats = load_features()[["quote_date", "vix_rank"]]

def by_regime(camp, name):
    t = camp.merge(feats, left_on="entry_date", right_on="quote_date", how="left")
    t["bucket"] = pd.qcut(t["vix_rank"], 3, labels=["low_vol", "mid_vol", "high_vol"])
    g = t.groupby("bucket", observed=True)["total_pnl"]
    return pd.DataFrame({f"{name}_avg_pnl": g.mean(), f"{name}_win": g.apply(lambda s: (s > 0).mean())})

pd.concat([by_regime(camps["no-roll"], "noroll"),
           by_regime(camps["defensive"], "def")], axis=1).round(2)

## Takeaways template

Fill these in after running the full 2010–2023 window:

1. **Best defensive trigger**: ___ (and is the neighborhood stable?)
2. **Offensive rolling**: does any threshold beat no-roll on P&L *per day in
   market*, not just total P&L?
3. **Regime dependence**: if defensive rolling only pays in high-vol buckets,
   arm it conditionally via the feature matrix.
4. Next step: combine the winner with notebook 03's entry filters and validate
   walk-forward (notebook 05 pattern) before trusting it.

Remember the caveats from the top: EOD triggers, mid fills, no costs. Add a
per-roll cost (e.g. $1–2 per contract + half the spread) before quoting any of
these numbers.

## Time roll: the tastylive variant
The practitioner canon rolls **at 21 DTE keeping the same strike** — a pure
time roll, no delta trigger. Compare against the delta-triggered variants
above **on P&L per day**, because time-rolling keeps campaigns in the market
~4-5x longer (max_rolls legs), which inflates total P&L mechanically.

In [ ]:
rows = []
for cfg in (RollConfig(),                              # exit at 21 DTE (no roll)
            RollConfig(time_roll_dte=21),              # tastylive time roll
            RollConfig(defensive_delta=0.50),          # delta-triggered
            RollConfig(defensive_delta=0.50, offensive_delta=0.12)):
    _, camp = simulate_campaigns(chains, cfg)
    rows.append(summarize_campaigns(camp, cfg.label()))
pd.DataFrame(rows).set_index("variant").round(2)

## Position sizing overlay
**Published finding** ([arXiv put-writing sizing study](https://arxiv.org/html/2508.16598v1)):
sizing policy changes the drawdown profile of put-writing more than most entry
tweaks. We post-process the *same* trade log with three sizing rules — entries
and exits identical, only contract counts differ:

- `fixed_1lot` — baseline
- `vix_scaled` — target_vix/VIX at entry (risk less when vol is high)
- `half_kelly` — fractional Kelly from a rolling window of *prior* trades only

Compare `return_over_maxdd` (MAR-style), not total P&L — sizing is a
risk-shaping tool, and a Kelly overlay that zeroes out most trades is telling
you the strategy's rolling edge estimate is weak, which is itself a finding.

In [ ]:
from lab.backtest import StrategyConfig, run_backtest, load_features
from lab.sizing import compare_sizing

flat = run_backtest(StrategyConfig(
    name="sizing_base", strategy="short_puts",
    params={"max_entry_dte": 45, "exit_dte": 21,
            "leg1_delta": {"min": 0.2, "target": 0.3, "max": 0.4}},
    start=START, end=END), chains=chains)
compare_sizing(flat.trade_log, load_features()).round(2)